# Inline Evals in Tracing: Score LLM Responses as Generated

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/inline-evals-tracing)

Score a Ridgeline support reply with the Agent Learning Kit and record the verdict and evaluation ID on the span that produced the reply.

## Install

Use a fresh Python 3.12 notebook kernel. The install cell uses the same Python interpreter as the notebook.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "fi-instrumentation-otel", "traceai-openai", "agent-learning-kit==0.3.7", "openai"], check=True);

### Credentials

Set environment variables before starting the kernel, or enter the keys at the hidden prompts. For self-hosting, set `FI_BASE_URL` to your API and `FI_TRACES_URL` to your collector before you register tracing.

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")

## Set up tracing and the Evaluator

Initialize the tracer, OpenAI instrumentation, and hosted evaluator once. `attach_eval` writes the result to the parent span; the instrumentor creates a separate child span for the model call.

In [ ]:
import os
import openai
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from fi.evals import Evaluator
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="ridgeline-inline-evals",
    set_global_tracer_provider=True,
    verbose=False,
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)


def attach_eval(span, *, eval_templates, inputs, model_name, custom_eval_name):
    batch = evaluator.evaluate(
        eval_templates=eval_templates, inputs=inputs, model_name=model_name,
    )
    check = batch.eval_results[0]
    if not check.eval_id or check.output is None:
        raise RuntimeError(f"No verdict for {custom_eval_name}: {check.reason}")
    prefix = f"quality.{custom_eval_name}"
    span.set_attribute(f"{prefix}.value", check.output)
    span.set_attribute(f"{prefix}.id", str(check.eval_id))
    print(f"{custom_eval_name}: {check.output}")


client = openai.OpenAI()
tracer = trace_provider.get_tracer(__name__)

## Attach a verdict to the answer span

Call `attach_eval` while your answer span is open. It records two attributes, `quality.groundedness_check.value` and `quality.groundedness_check.id`, before the span is exported.

In [ ]:
question = "What's the refund window for a final sale item?"

# Wrong chunk came back from retrieval: shipping info, not refund policy
context = "Standard shipping takes 3-5 business days within the continental US. Expedited orders arrive in 1-2 business days for an additional fee."

with tracer.start_as_current_span("answer-question") as span:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n{context}"},
            {"role": "user",   "content": question},
        ],
    )
    answer = response.choices[0].message.content

    span.set_attribute("raw.input",  question)
    span.set_attribute("raw.output", answer)

    # Run a groundedness check and attach it to this span
    attach_eval(
        span, eval_templates="groundedness",
        inputs={
            "input":   question,
            "output":  answer,
            "context": context,
        },
        model_name="turing_large",
        custom_eval_name="groundedness_check",  # name used in the span attributes
    )

print(f"Answer: {answer}")

# Flush spans before the script exits. BatchSpanProcessor buffers for up to 5 seconds
trace_provider.force_flush()

Replace the shipping passage with the actual refund policy, then ask the same question.

In [ ]:
context = "Orders can be refunded within 30 days of delivery if the item is unused and in original packaging. Final sale items are not eligible for refunds. Approved refunds are issued to the original payment method within 5-7 business days."

with tracer.start_as_current_span("answer-question") as span:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n{context}"},
            {"role": "user",   "content": question},
        ],
    )
    answer = response.choices[0].message.content

    span.set_attribute("raw.input",  question)
    span.set_attribute("raw.output", answer)

    attach_eval(
        span, eval_templates="groundedness",
        inputs={
            "input":   question,
            "output":  answer,
            "context": context,
        },
        model_name="turing_large",
        custom_eval_name="groundedness_check",
    )

print(f"Answer: {answer}")
trace_provider.force_flush()

## Run multiple evals on the same span

Pass the same open span to both checks. Their distinct names keep the toxicity verdict and instruction-adherence score from overwriting each other.

In [ ]:
user_input = "The customer says their refund hasn't arrived after 10 days. Draft a short, polite reply explaining our 5-7 business day refund window and the next step."

with tracer.start_as_current_span("explain-concept") as span:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": user_input}],
    )
    answer = response.choices[0].message.content

    span.set_attribute("raw.input",  user_input)
    span.set_attribute("raw.output", answer)

    # Check 1: Is the response toxicity-free?
    attach_eval(
        span, eval_templates="toxicity",
        inputs={"output": answer},
        model_name="turing_small",
        custom_eval_name="toxicity_check",
    )

    # Check 2: Did the response follow the prompt instructions?
    attach_eval(
        span, eval_templates="prompt_instruction_adherence",
        inputs={"output": answer, "prompt": user_input},
        model_name="turing_small",
        custom_eval_name="instruction_check",
    )

print(f"Answer: {answer}")

# Flush spans before the script exits. BatchSpanProcessor buffers for up to 5 seconds
trace_provider.force_flush()

## Inline evals on a full RAG pipeline

A realistic example: trace the full pipeline (retrieval and generation) and attach a groundedness eval to the generation span.

In [ ]:
from fi_instrumentation import using_user, using_session

def retrieve_docs(query: str) -> list[str]:
    # Simulate vector DB retrieval
    return [
        "Orders can be refunded within 30 days of delivery if the item is unused and in original packaging.",
        "Approved refunds are issued to the original payment method within 5-7 business days.",
    ]


def answer_question(question: str, user_id: str, session_id: str) -> str:
    with using_user(user_id), using_session(session_id):
        with tracer.start_as_current_span("rag-pipeline") as pipeline_span:
            pipeline_span.set_attribute("pipeline.question", question)

            # Retrieval span
            with tracer.start_as_current_span("retrieval") as ret_span:
                docs = retrieve_docs(question)
                ret_span.set_attribute("retrieval.doc_count", len(docs))

            # Generation span - eval attached here
            context = "\n".join(docs)
            with tracer.start_as_current_span("generation") as gen_span:
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": f"Answer from:\n{context}"},
                        {"role": "user",   "content": question},
                    ],
                )
                answer = response.choices[0].message.content
                gen_span.set_attribute("raw.output", answer)

                # Inline groundedness eval - did the answer stay grounded in the docs?
                attach_eval(
                    gen_span, eval_templates="groundedness",
                    inputs={
                        "input":   question,
                        "output":  answer,
                        "context": context,
                    },
                    model_name="turing_large",
                    custom_eval_name="groundedness_check",
                )

            return answer


result = answer_question(
    question="What's your refund window and how do I get my money back?",
    user_id="user-abc123",
    session_id="session-xyz789",
)
print(result)

trace_provider.force_flush()

## Find the results in Observe

Open **Tracing → ridgeline-inline-evals**. Inspect the `quality.*.value` and `quality.*.id` attributes on `answer-question`, `explain-concept`, or the `generation` span inside `rag-pipeline`. These are span attributes, not native Evals rows.